# Prompt Versioning

When accuracy moves, the first question is "what changed?" If prompts live as
inline string literals scattered across the code, you can't answer it. This
notebook shows the registry: prompts are version-pinned files, and the active
version is recorded on every call.

## Prompts are files, not literals

Each prompt lives at `prompts/{feature}/v{N}.txt`. A single dict,
`ACTIVE_VERSIONS`, pins which version is live per feature. `get_prompt` returns
`(version, template)` so the caller always knows which version it used.

In [ ]:
import sys
sys.path.insert(0, "../src")

from genai_prod_kit.prompts.registry import ACTIVE_VERSIONS, get_prompt

print("active versions:", ACTIVE_VERSIONS)
version, template = get_prompt("toy_summary")
print("\ntoy_summary version:", version)
print("template:", template)

## The version travels with the telemetry

Pass `prompt_version` into the Gateway and it lands in every `InvocationRecord`.
Now any logged call can be traced back to the exact prompt that produced it.

In [ ]:
import os, json
from genai_prod_kit.gateway import call_llm
from genai_prod_kit.providers.gemini import GeminiProvider
from genai_prod_kit.sinks.jsonl import JsonlSink

provider = GeminiProvider(os.environ["GEMINI_API_KEY"])
sink = JsonlSink("nb04_invocations.jsonl")

prompt = template.format(text="The 2026 FIFA World Cup will feature 48 national teams.")
call_llm(prompt, provider=provider, sink=sink, feature="toy_summary",
         model="gemini-2.5-flash", prompt_version=version)

last = json.loads(open("nb04_invocations.jsonl", encoding="utf-8").readlines()[-1])
print("recorded prompt_version:", last["prompt_version"])

## How you ship a prompt change

1. Add `prompts/{feature}/v2.txt` (keep v1 for history).
2. Flip `ACTIVE_VERSIONS[feature]` to `"v2"` and commit.
3. Re-run the eval (notebook 03). Accuracy is now attributable to v2 vs v1.

Because the version is in the telemetry, a regression points at a specific
prompt diff instead of a vague "something got worse."